## 1. Problem Statement & Objective

- **Problem statement:** Build a robust image classification model for the `Pestering-Data` dataset using a pretrained CNN backbone and transfer learning.
- **Objective:** Use a pretrained EfficientNetB0 backbone to leverage ImageNet features, fine-tune a small custom head, and evaluate performance. The notebook is intended to be reproducible and beginner-friendly.

## Dataset description

- The dataset is expected to be organized into class subdirectories (the standard `image_folder` format).
- The notebook will search for a local `Pestering-Data` folder (relative paths) and instruct the user if the dataset cannot be found.

## 2. Imports & Environment Setup

Required libraries are listed below. All are open-source. If any are missing, install them via `pip install` (example commands are provided).

In [ ]:
# Standard libraries
import os
import math
import datetime
import glob

# Data & ML
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision

# Useful utilities
from sklearn.metrics import confusion_matrix, classification_report

# Print versions for reproducibility
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Seaborn:', sns.__version__)

## 3. Dataset Loading & Exploration

- The notebook will try to auto-locate a `Pestering-Data` directory in the working directory or its parents.
- If not found, set `DATASET_DIR` to the dataset root (which must contain class subfolders).
- We use `image_dataset_from_directory` for safe, standard loading.

In [ ]:
# Attempt to locate the dataset folder automatically (non-destructive)
def find_dataset_dir(name='Pestering-Data', max_depth=4):
    cwd = os.getcwd()
    # search current and parent directories up to max_depth
    path = cwd
    for _ in range(max_depth+1):
        candidate = os.path.join(path, name)
        if os.path.isdir(candidate):
            return candidate
        path = os.path.dirname(path)
    # fallback: check common relative paths
    if os.path.isdir(name):
        return os.path.abspath(name)
    return None

DATASET_DIR = find_dataset_dir('Pestering-Data')
if DATASET_DIR is None:
    print('Dataset folder `Pestering-Data` not found automatically.')
    print('Please set DATASET_DIR to the dataset root (must contain class subfolders).')
    # Set a safe default variable which the user can edit before running further cells
    DATASET_DIR = './Pestering-Data'
else:
    print('Found dataset at:', DATASET_DIR)

# Show a preview of the directory structure if available
if os.path.isdir(DATASET_DIR):
    print('
Top-level entries in dataset directory:')
    for i, entry in enumerate(sorted(os.listdir(DATASET_DIR))[:50], start=1):
        print(f'{i:2d}.', entry)
else:
    print('
Dataset directory does not exist at', DATASET_DIR)

### Load images into a tf.data.Dataset

We use an 80/20 train/validation split. Images are resized to 224x224 (EfficientNetB0 input size can accept 224). Batch size is configurable.

In [ ]:
# Parameters - adjust as needed
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123
VAL_SPLIT = 0.2
AUTOTUNE = tf.data.AUTOTUNE

# Verify dataset path and load datasets if available
if os.path.isdir(DATASET_DIR):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=VAL_SPLIT,
        subset='training',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=VAL_SPLIT,
        subset='validation',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    class_names = train_ds.class_names
    num_classes = len(class_names)
    print(f'Found {num_classes} classes: {class_names}')
else:
    raise FileNotFoundError(f'Dataset directory not found at {DATASET_DIR}. Please set DATASET_DIR to the correct path.')

# Prefetch for performance
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

### Display sample images and class distribution

We show a few sample images from the training dataset and compute class counts.

In [ ]:
# Show a small grid of sample images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.suptitle('Sample training images')
plt.show()

# Compute class distribution from directory counts (robust)
class_counts = {}
for cls in class_names:
    pattern = os.path.join(DATASET_DIR, cls, '*')
    class_counts[cls] = len(glob.glob(pattern))

pd.Series(class_counts).sort_values().plot(kind='barh', figsize=(8, max(4, num_classes/2)))
plt.xlabel('Number of images')
plt.title('Class distribution')
plt.show()

## 4. Data Preprocessing

- We apply normalization matching the pretrained EfficientNet preprocessing.
- Add data augmentation layers during training to improve generalization.
- Keep augmentation only in the training pipeline.

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

# Data augmentation pipeline (applied only to training data)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.08),
], name='data_augmentation')

# Rescaling + pretrained model's preprocessing is applied in a mapping function
def preprocess_for_model(images, labels):
    images = tf.cast(images, tf.float32)
    images = effnet_preprocess(images)
    return images, labels

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(preprocess_for_model, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(preprocess_for_model, num_parallel_calls=AUTOTUNE)

## 5. Model Selection

- Chosen model: **EfficientNetB0** (available in `tf.keras.applications`).
- Rationale: EfficientNetB0 provides an excellent accuracy/compute tradeoff for transfer learning, is compact, and widely used in academic projects.
- We'll load ImageNet weights and use the convolutional base with `include_top=False`. Base layers will be frozen initially to train the classification head faster.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# Build the base model with pretrained ImageNet weights
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), pooling='avg')
base_model.trainable = False  # freeze the base for initial training
print('Base model loaded. Trainable layers:', sum([1 for l in base_model.layers if l.trainable]))

## 6. Model Architecture (Custom Head)

We add a small classification head on top of the pooled features from the base model. The head includes a dropout for regularization.

In [ ]:
def build_model(num_classes, base_model, dropout_rate=0.3):
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = inputs
    # Preprocessing (EfficientNet expects values preprocessed already)
    x = base_model(x, training=False)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='effnet_transfer')
    return model

model = build_model(num_classes=num_classes, base_model=base_model)
model.summary()

## 7. Training

- Loss: `sparse_categorical_crossentropy` (suitable for integer labels).
- Optimizer: Adam with a small learning rate.
- Epochs, batch size configurable. We include callbacks for early stopping and checkpointing.

In [ ]:
# Training parameters
EPOCHS = 10
LEARNING_RATE = 1e-4

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Callbacks
log_dir = os.path.join('logs', datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
callbacks = [
    keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy', mode='max'),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.TensorBoard(log_dir=log_dir)
]

# Train
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

## 8. Evaluation

We report final accuracy and plot loss/accuracy curves. We also compute a confusion matrix and classification report.

In [ ]:
# Plot training history
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='train_acc')
    axes[0].plot(history.history['val_accuracy'], label='val_acc')
    axes[0].set_title('Accuracy')
    axes[0].legend()
    # Loss
    axes[1].plot(history.history['loss'], label='train_loss')
    axes[1].plot(history.history['val_loss'], label='val_loss')
    axes[1].set_title('Loss')
    axes[1].legend()
    plt.show()

plot_history(history)

# Evaluate on validation set
val_loss, val_acc = model.evaluate(val_ds)
print(f'Validation loss: {val_loss:.4f}, Validation accuracy: {val_acc:.4f}')

### Confusion Matrix & Classification Report

We compute predictions on the validation set and build a confusion matrix. For large datasets this step may take time.

In [ ]:
# Gather all validation images and labels
y_true = []
y_pred = []

for images, labels in val_ds.unbatch().batch(BATCH_SIZE):
    preds = model.predict(images)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Classification report
print('Classification Report:
')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 9. Visualization of Sample Predictions

Display a few validation images with model predictions to inspect qualitative performance.

In [ ]:
def show_predictions(dataset, model, class_names, num_images=9):
    plt.figure(figsize=(10, 10))
    i = 1
    for images, labels in dataset.unbatch().batch(1).take(num_images):
        img = images[0]  # preprocessed image
        x = img.numpy()
        preds = model.predict(images)
        pred_label = class_names[np.argmax(preds[0])]
        true_label = class_names[int(labels[0].numpy())]
        ax = plt.subplot(3, 3, i)
        disp_img = x
        disp_img = (disp_img - disp_img.min()) / (disp_img.max() - disp_img.min())
        plt.imshow(disp_img)
        plt.title(f'True: {true_label}
Pred: {pred_label}')
        plt.axis('off')
        i += 1
    plt.tight_layout()
    plt.show()

show_predictions(val_ds, model, class_names, num_images=9)

## 10. Results & Conclusion

- **Final validation accuracy and loss** were printed above.
- **Observations:** The pretrained EfficientNetB0 backbone provides strong starting features; freezing the base speeds up initial convergence. Data augmentation improved robustness to small transformations.
- **Possible improvements:**
  - Unfreeze some top layers of the base model and fine-tune with a lower learning rate.
  - Use class-balanced sampling or weighting if class imbalance is significant.
  - Increase image resolution or use a larger EfficientNet variant for potentially better accuracy.

## 11. Reproducibility Notes

- How to run: place the `Pestering-Data` dataset folder in the workspace or set `DATASET_DIR` to its path. The folder must contain class subfolders (one per class) with images inside.
- Example dataset layout:
  - Pestering-Data/
    - class_A/
       - img001.jpg
    - class_B/
       - img010.jpg

- To run this notebook: ensure dependencies are installed (example):

```bash
pip install -U pip
pip install tensorflow matplotlib seaborn pandas scikit-learn
```

- Dataset path: Edit the `DATASET_DIR` variable near the "Dataset Loading" cell to point to your local dataset if auto-detection fails.
- Notes: This notebook does not use any private credentials or paid services. It uses open-source libraries only.

---

*Notebook created for Issue #94 — transfer learning with a pretrained model on Pestering-Data.*